# 15. XGB에서 stats 빼기 (H15)

## 배경
- 전처리 변경: `load_data()`에 변이 표기 정규화(X → *, fs 표기 통일, 이소폼 중복 제거) 추가 → 10 설정 그대로 CV 0.4113 → 0.4161
  - 정규화 후에도 test 환자당 변이 유전자 수는 train의 약 2배 (중앙값 14 vs 28) → stats 분포 차이가 남아 있음
- LR에서는 stats가 test 예측을 특정 아형으로 쏠리게 했음 (07 확인 1: LUAD 3.68배, SKCM 2.58배)
- XGB는 09부터 계속 any + stats로만 돌려서, XGB에서 stats가 LB에 이득인지 손해인지 따로 확인한 적이 없음

## 가설
- H15: test의 변이 수 차이 때문에 stats가 XGB에서도 LB를 깎는다 → stats를 빼면 LB가 오른다
- H0: XGB는 분기 구조라 stats의 분포 차이에 덜 흔들리므로, stats를 빼면 정보만 잃어 LB가 내려간다

## 검증 방법
- 모델: 10 최적 설정 고정 (depth 3, lr 0.1, 200그루, colsample 0.3)
- 데이터: `load_data()` (dedupe=True)
- 피처만 비교: any vs any + stats
- 두 버전 모두 제출 파일 저장
  - any + stats = "10 + 정규화" → 10(LB 0.3206)과 비교해 **정규화 효과** 확인
  - any만 → "10 + 정규화"와 비교해 **stats 효과** 확인
- 참고로 test 예측 아형 분포도 함께 기록 (쏠림 확인용, 판단 기준은 LB)

#### 데이터 준비, 비교, 제출 파일 저장
- `BalancedXGB`, `BEST`는 10·11과 동일
- 같은 fold(팀 공통 `run_cv`)로 두 피처 세트를 평가하고, 각각 제출 파일을 저장
- test 예측 분포: argmax로 고른 아형별 비율 상위 4개 (test 라벨은 쓰지 않음, 관찰용)

In [1]:
import sys; sys.path.append('..')
import pandas as pd, xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED
from src.data import load_data, get_gene_cols
from src.features import mutation_features
from src.validation import seed_everything, encode_target, run_cv, save_submission

seed_everything()
train, test, submission = load_data()   # dedupe=True (stats 개수 부풀림 방지)
genes = get_gene_cols(train)
y, le = encode_target(train)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)

class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

# 10에서 고른 최적 설정 (고정)
BEST = dict(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.3,
            tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)

SETS = {   # 이름: (train 피처, test 피처, 제출 파일 이름)
    'any+stats': (pd.concat([F_tr['any'], F_tr['stats']], axis=1),
                  pd.concat([F_te['any'], F_te['stats']], axis=1), 'xgb_d3_cs0.3_norm'),
    'any':       (F_tr['any'], F_te['any'], 'xgb_d3_cs0.3_norm_any_only'),
}

res, rows = {}, []
for name, (X, X_test, fname) in SETS.items():
    r = run_cv(lambda: BalancedXGB(**BEST), X, y, X_test, verbose=False)
    pred = pd.Series(le.inverse_transform(r['test_proba'].argmax(1))).value_counts(normalize=True)
    rows.append(dict(name=name, n_feat=X.shape[1], cv=round(r['cv_mean'], 4), std=round(r['cv_std'], 4),
                     test_pred_top4=pred.head(4).round(3).to_dict()))
    print(rows[-1])
    save_submission(submission, r['test_proba'], le, name=fname, cv_score=r['cv_mean'])
    res[name] = r

res15 = pd.DataFrame(rows)
res15

{'name': 'any+stats', 'n_feat': 4393, 'cv': 0.4161, 'std': 0.006, 'test_pred_top4': {'STES': 0.126, 'COAD': 0.124, 'TGCT': 0.051, 'LUSC': 0.051}}
저장 완료: C:\Users\USER\Desktop\hackathon\submissions\1110_xgb_d3_cs0.3_norm_0.416.csv
{'name': 'any', 'n_feat': 4384, 'cv': 0.3579, 'std': 0.0059, 'test_pred_top4': {'COAD': 0.221, 'THYM': 0.091, 'BRCA': 0.062, 'OV': 0.058}}
저장 완료: C:\Users\USER\Desktop\hackathon\submissions\1110_xgb_d3_cs0.3_norm_any_only_0.358.csv


,name,n_feat,cv,std,test_pred_top4
0,any+stats,4393,0.4161,0.0060,"{'STES': 0.126, 'COAD': 0.124, 'TGCT': 0.051, ..."
1,any,4384,0.3579,0.0059,"{'COAD': 0.221, 'THYM': 0.091, 'BRCA': 0.062, ..."


## 결과

| 피처 | 열 수 | CV (± std) | test 예측 상위 | LB |
| :--- | --: | :-: | :--- | :-: |
| any + stats (= 10 + 정규화) | 4,393 | 0.4161 ± 0.0060 | STES 12.6%, COAD 12.4%, TGCT 5.1%, LUSC 5.1% | |
| any | 4,384 | 0.3579 ± 0.0059 | **COAD 22.1%**, THYM 9.1%, BRCA 6.2%, OV 5.8% | |

- 참고: 10 (정규화 전, any + stats) CV 0.4113 ± 0.0024 / LB 0.3206
- any + stats CV 0.4161 ± 0.0060은 11 기준선과 동일 → 같은 조건 재현 확인
- stats를 빼면 CV −0.058 (std의 약 10배) → XGB에서 stats는 CV에 매우 큰 비중을 차지하는 피처
- stats를 빼면 test 예측이 COAD로 쏠림 (22.1%, train COAD 비율 3.6%)
  - 처음 LR(any만)에서 본 COAD 쏠림과 같은 모양 → stats가 any만으로 생기는 쏠림을 막아주는 역할도 하는 것으로 보임
  - test 라벨 없이 본 예측 분포라 판단 기준은 아님 (관찰용)

## 결론
- 정규화 효과 (any + stats vs 10): CV 0.4113 → 0.4161 (+0.005), LB (제출 후 기입)
- stats 효과 (any vs any + stats): CV −0.058, LB (제출 후 기입)
- H15 채택 / 기각: (LB 비교 후 기입)
  - any만의 LB가 any + stats보다 높으면 H15 채택 → stats의 분포 차이가 CV 이득보다 큰 손해
  - 낮으면 H15 기각 → XGB에서는 stats가 test에서도 이득

## 다음
-